### Import Dependencies

In [22]:
import pandas as pd
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

/Users/ganesh.m/Projects/ai-engineering-bootcamp/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read the sampled dataset with Amazon inventory data

In [ ]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
df_items.head()

In [ ]:
list(df_items["features"].items())[0]

In [ ]:
list(df_items["images"].items())[0]

### Preprocess title and features

In [23]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [24]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [25]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [26]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...","Bosttor Bluetooth Beanie Hat with Light, Headl...",26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51b7qcj8dZ...
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41WsGRr-3T...
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41fsjaknf1...
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51YFypeuZh...
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41V7Wi2ckb...


In [27]:
list(df_items["description"].items())[0]

(0,
 'Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It point

In [28]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_.jpg')

### Sample 50 items from the dataset

In [29]:
df_sample = df_items.sample(50, random_state=42)

In [30]:
data_to_embed = df_sample[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

In [31]:
data_to_embed

[{'description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitting on to allow

### Define the local embedding function

Uses Sentence Transformers without an API key or API credits. Install dependencies with `uv sync` from the project root. The first run downloads the model; later runs use the cached model. Inputs longer than 256 word pieces are truncated, so chunk longer descriptions before embedding.


In [33]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Downloads the model on first use; embeddings are computed locally.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
EMBEDDING_DIMENSION = embedding_model.get_embedding_dimension()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8089.99it/s]


In [34]:
embedding = embedding_model.encode("Random text", normalize_embeddings=True).tolist()

In [35]:
len(embedding)  # 384 dimensions

384

In [36]:
def get_embedding(text):
    return embedding_model.encode(text, normalize_embeddings=True).tolist()

In [37]:
get_embedding("Hi.")

[-0.102968730032444,
 0.04557419568300247,
 -0.007624595891684294,
 0.05659149959683418,
 -0.01636890508234501,
 0.009336107410490513,
 0.03302436321973801,
 0.013783039525151253,
 -0.08098834753036499,
 -0.05617319419980049,
 -0.006675345823168755,
 0.03724903613328934,
 0.004329139366745949,
 -0.03708263859152794,
 0.07661013305187225,
 0.039964497089385986,
 -0.04627358168363571,
 -0.10069533437490463,
 -0.09666068851947784,
 -0.011142770759761333,
 0.012282272800803185,
 0.01949329860508442,
 -0.006779369432479143,
 0.012672154232859612,
 -0.019425740465521812,
 -0.03090687468647957,
 0.019271591678261757,
 0.08563010394573212,
 -0.08745323866605759,
 -0.025416268035769463,
 0.06312933564186096,
 0.07879933714866638,
 -0.04927695542573929,
 -0.014727417379617691,
 -0.03007504716515541,
 0.054524339735507965,
 -0.08387336879968643,
 -0.12049472332000732,
 0.020348655059933662,
 0.008122068829834461,
 0.013807752169668674,
 0.021813763305544853,
 0.002873913384974003,
 0.038208581507

### Create Qdrant Collection

In [38]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [39]:
# Keep local embeddings separate from existing OpenAI vectors.
COLLECTION_NAME = "Amazon-items-minilm-l6-v2"

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=EMBEDDING_DIMENSION, distance=Distance.COSINE
        ),
    )

### Embed Data

#### Test

In [40]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Test Text"),
    payload={
        "text": "Test Text",
        "model": EMBEDDING_MODEL_NAME,
    }
)

In [41]:
pointstruct

PointStruct(id=0, vector=[0.03579318895936012, 0.08075191080570221, -0.022894637659192085, 0.03583970293402672, -0.01652580499649048, 0.023141738027334213, 0.06864473223686218, 0.013322373852133751, 0.018502287566661835, -0.01246105507016182, 0.07817531377077103, -0.012404761277139187, 0.035567961633205414, 0.007992557249963284, 0.03585612028837204, -0.03761390596628189, 0.04076765105128288, -0.017905525863170624, -0.03391684964299202, -0.05569084361195564, 0.02311631292104721, 0.04743530973792076, 0.017255915328860283, 0.04447339102625847, -0.01573190465569496, 0.1007293090224266, -0.08943485468626022, 0.02642364799976349, 0.026291098445653915, -0.02133486047387123, -0.021611837670207024, 0.03308771178126335, 0.07231834530830383, 0.0723659098148346, 0.057041022926568985, -0.0033079979475587606, -0.03990708291530609, 0.031113270670175552, 0.017501402646303177, 0.0364811047911644, -0.06317992508411407, -0.11546610295772552, 0.03893391788005829, 0.02778879553079605, 0.09715814143419266, 

### Amazon data

In [42]:
data_to_embed

[{'description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitting on to allow

In [43]:
pointstructs = []
for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data["description"])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )

In [44]:
pointstructs

[PointStruct(id=0, vector=[0.0724378228187561, -0.016690926626324654, 0.04180918261408806, -0.010281487368047237, 0.0762273520231247, -0.028815966099500656, -0.03298867493867874, 0.008274232037365437, 0.038386713713407516, 0.06850682944059372, 0.010297921486198902, 0.05341999977827072, 0.003835461102426052, -0.062011659145355225, 0.02743036113679409, 0.0947190374135971, 0.018968701362609863, 0.007845745421946049, -0.05774394050240517, 0.0027199338655918837, 0.031245172023773193, -0.02074156142771244, -0.0344129353761673, -0.018212424591183662, -0.09499260783195496, -0.00454118475317955, 0.05315660685300827, -0.012347904033958912, -0.06563340872526169, -0.017937881872057915, 0.004003033973276615, -0.043507251888513565, -0.05690225586295128, 0.03175600245594978, -0.047959934920072556, -0.03975198417901993, -0.03657187148928642, -0.03670172765851021, -0.06245850399136543, -0.011591298505663872, -0.007074819877743721, -0.08210450410842896, 0.10037454217672348, -0.022510739043354988, -0.080

In [45]:
len(pointstructs)

50

### Write embedded data to Qdrant

In [46]:
qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=pointstructs,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Define a function for data retrieval

In [47]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=k
    )
    return results

In [48]:
retrieve_data("Do you have anything that can play vinyls", k=10).points

[ScoredPoint(id=29, version=1, score=0.5452032, payload={'description': 'LP&No.1 Record Player with External Speakers, 3 Speed Vintage Belt-Drive Vinyl Turntable with Bluetooth Playback & Auto-Stop （Light Blue） 【3-Speed Vinyl Record player】 The diamond-tipped stylus plays all of 33 1/3, 45, and 78 RPMs records. Auto-stop setting makes it possible to stop automatically when it reaches the end of the record 【Stereo Turntable with Outstanding Sound】Equipped with a separable pair of speakers, let you enjoy the full,real and live vinyl audio in dual speakers easily.The volume is loud enough for many occasions 【Quality Vinyl Player and Speaker Set】Dual Bookshelf speakers allows for multiple placement options. You can make the speakers as a powerful sound base, or place it as a bookshelf or record shelf 【Modern Turntable with Bluetooth】Great record player as well as ability to work as a Bluetooth speaker.Turn to BT on your phone and hook up to any Bluetooth device easily for premium sound 【Wo